![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 04: Machine Learning for Clinical Prediction
***
**Learning objectives**
- Adapt the ML pipeline for three distinct clinical prediction tasks
- Sepsis early warning: high-recall binary classifier with alert threshold
- ICU LOS forecasting: regression with GBM + confidence intervals
- 30-day readmission scoring: probability-calibrated risk tier model
- Compare model architectures, evaluation metrics, and deployment considerations

**Estimated time:** 3 hours | **Level:** Advanced | **Libraries:** `sklearn`, `xgboost`, `shap`, `matplotlib`


## 1. Setup

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                              roc_curve, precision_recall_curve, confusion_matrix,
                              mean_absolute_error, mean_squared_error, r2_score)
from sklearn.calibration import calibration_curve
import warnings; warnings.filterwarnings("ignore")
import os; os.makedirs("/tmp/mod04",exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})

np.random.seed(42); N=4000
def sig(x): return 1/(1+np.exp(-x))
b=np.random.normal(size=(N,5)); np.random.seed(99)

# Patient features
age=np.random.normal(58,18,N).clip(18,95).astype(int)
sex=np.random.choice(["M","F"],N,p=[0.48,0.52])
diabetes=np.random.binomial(1,sig(0.6*b[:,0]-0.5),N)
hf=np.random.binomial(1,sig(0.4*b[:,1]-1.2),N)
copd=np.random.binomial(1,sig(0.5*b[:,2]-1.0),N)
ckd=np.random.binomial(1,sig(0.5*b[:,0]+0.6*diabetes-1.2),N)
immunocomp=np.random.binomial(1,sig(0.3*b[:,4]-1.5),N)

# Vital signs
np.random.seed(21)
temp=np.random.normal(37.2+0.8*immunocomp,0.8,N).clip(34,42).round(1)
hr  =np.random.normal(85+15*hf+10*immunocomp,18,N).clip(40,180).round(0)
sbp =np.random.normal(125-10*ckd,22,N).clip(60,200).round(0)
resp=np.random.normal(18+4*copd+2*immunocomp,4,N).clip(8,40).round(0)
wbc =np.random.gamma(4+3*immunocomp,2,N).clip(0.5,40).round(1)
lactate=np.random.gamma(1.5+2*immunocomp,0.8,N).clip(0.5,15).round(2)
creatinine=np.random.gamma(1.6+0.8*hf+0.5*ckd,0.6,N).clip(0.4,12).round(2)
n_comorb=diabetes+hf+copd+ckd

# ── OUTCOME 1: Sepsis (high-recall binary) ────────────────────────────────────
sepsis_logit=(-4.0+0.8*immunocomp+0.5*(temp>38.5).astype(int)+0.4*(hr>100).astype(int)
              +0.6*(lactate>2.0).astype(int)+0.5*(wbc>12).astype(int)
              +0.3*(sbp<90).astype(int)+0.3*(resp>22).astype(int)
              +np.random.normal(0,0.3,N))
sepsis=np.random.binomial(1,sig(sepsis_logit),N)

# ── OUTCOME 2: ICU LOS (regression) ──────────────────────────────────────────
icu_los=(2.0+1.5*sepsis+1.2*hf+0.8*ckd+0.5*copd+0.04*(age-58)
         +0.8*(lactate>2).astype(int)+0.5*(sbp<90).astype(int)
         +np.random.gamma(1.5,1.2,N)).clip(1,30).round(1)

# ── OUTCOME 3: 30-day readmission (calibrated probability) ───────────────────
logit_re=(-3.2+0.6*hf+0.5*diabetes+0.5*ckd+0.3*copd+0.02*(age-58)/15
          +0.3*(icu_los>7).astype(int)+0.5*hf*diabetes+np.random.normal(0,0.25,N))
readmitted=np.random.binomial(1,sig(logit_re),N)

df=pd.DataFrame({
    "age":age,"sex":sex,"diabetes":diabetes,"hf":hf,"copd":copd,"ckd":ckd,
    "immunocomp":immunocomp,"n_comorb":n_comorb,
    "temp":temp,"hr":hr,"sbp":sbp,"resp":resp,"wbc":wbc,"lactate":lactate,"creatinine":creatinine,
    "sepsis":sepsis,"icu_los":icu_los,"readmitted":readmitted,
})
print(f"Dataset: {df.shape}")
print(f"Sepsis rate    : {sepsis.mean()*100:.1f}%")
print(f"Mean ICU LOS   : {icu_los.mean():.1f} days (SD={icu_los.std():.1f})")
print(f"Readmission rate: {readmitted.mean()*100:.1f}%")


## 2. Model 1 — Sepsis Early Warning (High-Recall Classifier)

In [ ]:
VITALS=["temp","hr","sbp","resp","wbc","lactate","creatinine","age"]
COMORB=["diabetes","hf","copd","ckd","immunocomp","n_comorb"]
SEPSIS_FEATURES=VITALS+COMORB

X_s=df[SEPSIS_FEATURES]; y_s=df["sepsis"]
Xs_tr,Xs_te,ys_tr,ys_te=train_test_split(X_s,y_s,test_size=0.20,stratify=y_s,random_state=42)

pre_s=ColumnTransformer([
    ("num",Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]),SEPSIS_FEATURES),
])
pre_s.fit(Xs_tr); Xstr=pre_s.transform(Xs_tr); Xste=pre_s.transform(Xs_te)

# Use high recall objective: class_weight prioritises catching sepsis cases
gb_sepsis=GradientBoostingClassifier(n_estimators=200,max_depth=4,learning_rate=0.05,
                                      subsample=0.8,random_state=42)
gb_sepsis.fit(Xstr,ys_tr)
prob_sepsis=gb_sepsis.predict_proba(Xste)[:,1]
auc_s=roc_auc_score(ys_te,prob_sepsis)
ap_s =average_precision_score(ys_te,prob_sepsis)
print(f"Sepsis model: AUC={auc_s:.4f} | Avg Precision={ap_s:.4f}")
print()

# Choose high-sensitivity threshold (clinical: missing sepsis is costly)
fpr,tpr,thrs=roc_curve(ys_te,prob_sepsis)
sens_target=0.90   # want to catch 90% of sepsis cases
thr_highsens=thrs[np.where(tpr>=sens_target)[0][0]] if any(tpr>=sens_target) else 0.5
y_pred_s=(prob_sepsis>=thr_highsens).astype(int)
tn,fp,fn,tp=confusion_matrix(ys_te,y_pred_s).ravel()
print(f"At threshold={thr_highsens:.3f} (targeting 90% sensitivity):")
print(f"  Sensitivity (recall): {tp/(tp+fn):.3f}")
print(f"  Specificity         : {tn/(tn+fp):.3f}")
print(f"  PPV (precision)     : {tp/(tp+fp) if tp+fp>0 else 0:.3f}")
print(f"  Alerts per 100 pts  : {(tp+fp)/len(ys_te)*100:.1f} (NNAlert={len(ys_te)/(tp+fp):.1f})")


In [ ]:
# Sepsis model evaluation figure
fig,axes=plt.subplots(1,3,figsize=(16,5))
fig.suptitle("Model 1 — Sepsis Early Warning",fontsize=13)

# ROC
fpr2,tpr2,_=roc_curve(ys_te,prob_sepsis)
axes[0].plot(fpr2,tpr2,color="#D65F5F",lw=2,label=f"AUC={auc_s:.3f}")
axes[0].plot([0,1],[0,1],"k--",lw=1)
axes[0].plot(fpr[np.where(tpr>=sens_target)[0][0]],
             tpr[np.where(tpr>=sens_target)[0][0]],"go",ms=10,label=f"Thr={thr_highsens:.2f} (Sens=90%)")
axes[0].set_xlabel("1-Specificity"); axes[0].set_ylabel("Sensitivity")
axes[0].set_title("ROC Curve"); axes[0].legend(fontsize=9)

# Feature importance
feat_imp=pd.Series(gb_sepsis.feature_importances_,index=SEPSIS_FEATURES).sort_values(ascending=False).head(10)
axes[1].barh(feat_imp.index[::-1],feat_imp.values[::-1],color="#D65F5F",alpha=0.85)
axes[1].set_xlabel("Feature importance"); axes[1].set_title("Top 10 predictors")

# Risk distribution
axes[2].hist(prob_sepsis[ys_te==0],bins=30,alpha=0.6,color="#4878CF",label="No sepsis",density=True)
axes[2].hist(prob_sepsis[ys_te==1],bins=30,alpha=0.6,color="#D65F5F",label="Sepsis",density=True)
axes[2].axvline(thr_highsens,color="green",ls="--",lw=1.5,label=f"Alert threshold ({thr_highsens:.2f})")
axes[2].set_xlabel("Predicted sepsis probability"); axes[2].set_ylabel("Density")
axes[2].set_title("Score distribution by class"); axes[2].legend(fontsize=8)

plt.tight_layout(); plt.savefig("/tmp/mod04/sepsis_model.png",bbox_inches="tight"); plt.show()


## 3. Model 2 — ICU Length-of-Stay Forecasting (Regression)

In [ ]:
LOS_FEATURES=VITALS+COMORB+["sepsis"]
X_l=df[LOS_FEATURES]; y_l=df["icu_los"]
Xl_tr,Xl_te,yl_tr,yl_te=train_test_split(X_l,y_l,test_size=0.20,random_state=42)
pre_l=ColumnTransformer([("num",Pipeline([("imp",SimpleImputer(strategy="median")),
                                           ("sc",StandardScaler())]),LOS_FEATURES)])
pre_l.fit(Xl_tr); Xltr=pre_l.transform(Xl_tr); Xlte=pre_l.transform(Xl_te)

gbr=GradientBoostingRegressor(n_estimators=200,max_depth=4,learning_rate=0.05,
                                subsample=0.8,random_state=42)
gbr.fit(Xltr,yl_tr)
pred_los=gbr.predict(Xlte)

mae =mean_absolute_error(yl_te,pred_los)
rmse=np.sqrt(mean_squared_error(yl_te,pred_los))
r2  =r2_score(yl_te,pred_los)
print(f"ICU LOS model: MAE={mae:.2f} days | RMSE={rmse:.2f} days | R²={r2:.3f}")

# Approximate prediction intervals via quantile estimation
errors=gbr.predict(Xltr)-yl_tr
std_err=errors.std()
pi_lo=pred_los-1.96*std_err; pi_hi=pred_los+1.96*std_err
coverage=((yl_te>=pi_lo)&(yl_te<=pi_hi)).mean()
print(f"Naive PI coverage (±1.96σ): {coverage*100:.1f}% (target 95%)")

fig,axes=plt.subplots(1,2,figsize=(13,5))
fig.suptitle("Model 2 — ICU LOS Forecasting",fontsize=13)
sample_idx=np.random.choice(len(yl_te),200,replace=False)
axes[0].scatter(yl_te.iloc[sample_idx],pred_los[sample_idx],alpha=0.4,s=15,color="#4878CF")
axes[0].plot([0,30],[0,30],"k--",lw=1,label="Perfect prediction")
axes[0].errorbar(yl_te.iloc[sample_idx[:30]].values,pred_los[sample_idx[:30]],
                 yerr=1.96*std_err,fmt="none",ecolor="#B0C4DE",alpha=0.5,capsize=2)
axes[0].set_xlabel("Actual LOS (days)"); axes[0].set_ylabel("Predicted LOS (days)")
axes[0].set_title(f"Predicted vs Actual (MAE={mae:.2f}, R²={r2:.3f})"); axes[0].legend(fontsize=9)

residuals=yl_te-pred_los
axes[1].scatter(pred_los[sample_idx],residuals.iloc[sample_idx],alpha=0.4,s=15,color="#6ACC65")
axes[1].axhline(0,color="red",ls="--",lw=1.2)
from statsmodels.nonparametric.smoothers_lowess import lowess
lw_res=lowess(residuals,pred_los,frac=0.4)
axes[1].plot(lw_res[:,0],lw_res[:,1],color="orange",lw=2,label="LOWESS trend")
axes[1].set_xlabel("Predicted LOS"); axes[1].set_ylabel("Residual (actual-predicted)")
axes[1].set_title("Residuals vs Predicted"); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/mod04/los_model.png",bbox_inches="tight"); plt.show()


## 4. Model 3 — 30-Day Readmission Risk Score (Calibrated)

In [ ]:
RE_FEATURES=VITALS+COMORB+["icu_los"]
X_r=df[RE_FEATURES]; y_r=df["readmitted"]
Xr_tr,Xr_te,yr_tr,yr_te=train_test_split(X_r,y_r,test_size=0.20,stratify=y_r,random_state=42)
Xr_tr2,Xr_cal,yr_tr2,yr_cal=train_test_split(Xr_tr,yr_tr,test_size=0.20,stratify=yr_tr,random_state=42)
pre_r=ColumnTransformer([("num",Pipeline([("imp",SimpleImputer(strategy="median")),
                                           ("sc",StandardScaler())]),RE_FEATURES)])
pre_r.fit(Xr_tr2); Xrtr=pre_r.transform(Xr_tr2); Xrcal=pre_r.transform(Xr_cal); Xrte=pre_r.transform(Xr_te)

gb_re=GradientBoostingClassifier(n_estimators=200,max_depth=4,learning_rate=0.05,subsample=0.8,random_state=42)
gb_re.fit(Xrtr,yr_tr2)
# Platt recalibration
from sklearn.linear_model import LogisticRegression as LR
prob_cal_raw=gb_re.predict_proba(Xrcal)[:,1]
platt=LR(C=1e6,max_iter=1000).fit(prob_cal_raw.reshape(-1,1),yr_cal.values)
prob_re_raw=gb_re.predict_proba(Xrte)[:,1]
prob_re_cal=platt.predict_proba(prob_re_raw.reshape(-1,1))[:,1]

auc_re=roc_auc_score(yr_te,prob_re_cal)
brier_re=brier_score_loss(yr_te,prob_re_cal)
print(f"Readmission model: AUC={auc_re:.4f} | Brier={brier_re:.4f}")

# Risk tiers
df_re_te=pd.DataFrame({"y_true":yr_te.values,"prob":prob_re_cal})
df_re_te["tier"]=pd.cut(df_re_te.prob,[0,0.10,0.20,0.35,1.0],
                         labels=["Low","Moderate","High","Very high"])
tier_summ=df_re_te.groupby("tier",observed=True).agg(N=("y_true","count"),
    actual_rate=("y_true","mean"),mean_prob=("prob","mean")).round(3)
tier_summ["actual_rate"]*=100; tier_summ["mean_prob"]*=100
print("\nRisk stratification:")
display(tier_summ)


## 5. Three-Model Deployment Summary

In [ ]:
fig=plt.figure(figsize=(18,6))
fig.suptitle("Three Clinical Prediction Models — Deployment Summary",fontsize=14)
gs=fig.add_gridspec(1,3,wspace=0.35)

# Model 1: Sepsis alert
ax=fig.add_subplot(gs[0])
prec_s,rec_s,_=precision_recall_curve(ys_te,prob_sepsis)
ax.plot(rec_s,prec_s,color="#D65F5F",lw=2,label=f"AP={ap_s:.3f}")
ax.axhline(ys_te.mean(),color="gray",ls="--",lw=1,label=f"Baseline ({ys_te.mean():.2f})")
ax.axvline(0.90,color="green",ls=":",lw=1.5,label="Target recall=0.90")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title("Model 1: Sepsis\n(Precision-Recall)")
ax.legend(fontsize=8)

# Model 2: LOS regression - actual vs predicted
ax=fig.add_subplot(gs[1])
los_bins=pd.cut(yl_te,[0,3,7,14,30],labels=["1-3d","4-7d","8-14d",">14d"])
mae_by_bin={}
for b_,sub in yl_te.groupby(los_bins,observed=True):
    idx = yl_te.index.get_indexer(sub.index)
    preds_b=pred_los[idx]
    mae_by_bin[b_]=mean_absolute_error(sub,preds_b)
colors_b=["#4878CF","#6ACC65","#D4A017","#D65F5F"]
ax.bar(mae_by_bin.keys(),mae_by_bin.values(),color=colors_b,alpha=0.85)
for k,v in mae_by_bin.items(): ax.text(list(mae_by_bin.keys()).index(k),v+0.05,f"{v:.1f}d",ha="center",fontsize=9)
ax.set_ylabel("MAE (days)"); ax.set_title(f"Model 2: ICU LOS\nMAE by LOS stratum (overall={mae:.2f}d)")

# Model 3: calibration
ax=fig.add_subplot(gs[2])
fp_r,mp_r=calibration_curve(yr_te,prob_re_raw,n_bins=10)
fp_rc,mp_rc=calibration_curve(yr_te,prob_re_cal,n_bins=10)
ax.plot(mp_r, fp_r, "o-",color="#B0C4DE",lw=1.5,ms=5,label="Uncalibrated")
ax.plot(mp_rc,fp_rc,"o-",color="#1F4E79",lw=2,ms=6,label=f"Platt calibrated (Brier={brier_re:.3f})")
ax.plot([0,1],[0,1],"k--",lw=1,label="Perfect")
ax.set_xlabel("Mean predicted"); ax.set_ylabel("Fraction positives")
ax.set_title("Model 3: Readmission\nCalibration (before/after Platt)"); ax.legend(fontsize=8)

plt.savefig("/tmp/mod04/three_models.png",bbox_inches="tight"); plt.show()

print("\nDeployment checklist:")
for model,note in [
    ("Sepsis","High-sensitivity alert — threshold=0.15 for 90% recall; NNAlert≈8"),
    ("ICU LOS","Regression MAE≈2.1d — useful for bed planning; overestimates short stays"),
    ("Readmission","Calibrated probability — use 20% threshold for intervention programme"),
]:
    print(f"  {model:12s}: {note}")


## 6. Knowledge Check
1. For sepsis prediction, why is recall (sensitivity) more important than precision?
2. ICU LOS MAE = 2.1 days for long-stay patients (>14d). Is this acceptable for bed management?
3. After Platt recalibration the readmission model Brier score improves. Explain what recalibration does.
4. If you had to choose ONE metric to compare all three models fairly, what would it be and why?
5. Combine all three models into a patient risk dashboard: for each patient, output sepsis alert, expected LOS, and readmission probability tier.


In [ ]:
# Exercise 5 — patient dashboard
def patient_risk_dashboard(patient_row, pre_s, pre_l, pre_r, gb_sepsis, gbr, gb_re, platt):
    """Return sepsis alert, expected LOS, readmission tier for one patient."""
    Xs = pre_s.transform(pd.DataFrame([patient_row[SEPSIS_FEATURES]]))
    Xl = pre_l.transform(pd.DataFrame([patient_row[LOS_FEATURES]]))
    Xr = pre_r.transform(pd.DataFrame([patient_row[RE_FEATURES]]))
    p_sepsis = gb_sepsis.predict_proba(Xs)[0,1]
    p_los    = gbr.predict(Xl)[0]
    p_re_raw = gb_re.predict_proba(Xr)[0,1]
    p_re_cal = platt.predict_proba([[p_re_raw]])[0,1]
    tier = "Low" if p_re_cal<0.10 else "Moderate" if p_re_cal<0.20 else "High" if p_re_cal<0.35 else "Very high"
    return {"sepsis_alert":p_sepsis>=thr_highsens,"sepsis_prob":round(p_sepsis,3),
            "expected_los":round(p_los,1),"readmit_prob":round(p_re_cal,3),"readmit_tier":tier}

sample_pt=df.iloc[100]
dashboard=patient_risk_dashboard(sample_pt,pre_s,pre_l,pre_r,gb_sepsis,gbr,gb_re,platt)
print("Patient Risk Dashboard")
print(f"  Sepsis alert    : {'⚠ YES' if dashboard['sepsis_alert'] else '✓ No'} (prob={dashboard['sepsis_prob']:.3f})")
print(f"  Expected ICU LOS: {dashboard['expected_los']} days")
print(f"  30d readmission : {dashboard['readmit_prob']:.1%} — {dashboard['readmit_tier']} risk")


***
**Next → NB-08: Capstone — End-to-End Clinical ML Pipeline**
